## Практична

## завдання 1

In [1]:
from collections import Counter
import re

qa_pairs = [
    ("How can I reset my password?", "You can reset it through the account settings."),
    ("Where is my order?", "You can track your order in the purchases section."),
    ("What are your working hours?", "We work from nine AM to six PM every weekday."),
    ("How do I cancel my subscription?", "Go to billing and click cancel plan."),
    ("Do you offer international shipping?", "Yes we ship to over fifty countries worldwide."),
    ("Can I change my email address?", "Yes update your email in profile settings."),
    ("What payment methods do you accept?", "We accept credit cards and PayPal."),
    ("How long does delivery take?", "Standard delivery usually takes three to five days."),
    ("Is there a free trial available?", "Yes we offer a fourteen day free trial."),
    ("How can I contact customer support?", "You can email us or use the live chat."),
    ("Where can I find my invoices?", "All invoices are stored in your billing tab."),
    ("Do you offer refunds?", "We provide full refunds within thirty days.")
]

def clean_text(text):
  text = text.lower()
  text = re.sub(r"[^\w\s]", "", text)
  return text.strip()

cleaned_pairs = [(clean_text(q), clean_text(a)) for q, a in qa_pairs]

special_tokens = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

input_tokens = []
target_tokens = []

for q, a in cleaned_pairs:
  q_tok = q.split()
  a_tok = ["<SOS>"] + a.split() + ["<EOS>"]

  input_tokens.append(q_tok)
  target_tokens.append(a_tok)

all_words = [word for seq in (input_tokens + target_tokens) for word in seq if word not in special_tokens]
word_counts = Counter(all_words)

vocab = {token: idx for idx, token in enumerate(special_tokens)}
for word in word_counts:
  vocab[word] = len(vocab)

idx_to_token = {idx: token for token, idx in vocab.items()}

max_input_len = max(len(seq) for seq in input_tokens)
max_target_len = max(len(seq) for seq in target_tokens)

print(f"Розмір словника: {len(vocab)} токенів")
print(f"Приклад словника (перші 10): {list(vocab.items())[:10]}")
print(f"Максимальна довжина запитання: {max_input_len} токенів")
print(f"Максимальна довжина відповіді: {max_target_len} токенів")
print("\nПриклад обробленої пари:")
print("Вхідні токени:", input_tokens[0])
print("Вихідні токени:", target_tokens[0])

Розмір словника: 101 токенів
Приклад словника (перші 10): [('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('how', 4), ('can', 5), ('i', 6), ('reset', 7), ('my', 8), ('password', 9)]
Максимальна довжина запитання: 6 токенів
Максимальна довжина відповіді: 12 токенів

Приклад обробленої пари:
Вхідні токени: ['how', 'can', 'i', 'reset', 'my', 'password']
Вихідні токени: ['<SOS>', 'you', 'can', 'reset', 'it', 'through', 'the', 'account', 'settings', '<EOS>']


## завдання 2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

max_len = max(max_input_len, max_target_len)

def encode_and_pad(tokens_list, vocab, max_len):
  encoded = []
  for seq in tokens_list:
    # Перетворюємо слова на індекси
    indices = [vocab.get(token, vocab["<UNK>"]) for token in seq]
    # Доповнюємо <PAD> до max_len
    padded = indices + [vocab["<PAD>"]] * (max_len - len(indices))
    encoded.append(padded)
  return torch.tensor(encoded, dtype=torch.long)

X_indices = encode_and_pad(input_tokens, vocab, max_len)
Y_indices = encode_and_pad(target_tokens, vocab, max_len)

vocab_size = len(vocab)

X_one_hot = F.one_hot(X_indices, num_classes=vocab_size).float()

class SimpleQARNN(nn.Module):

  def __init__(self, input_size, hidden_size, output_size, num_layers=1):
    super(SimpleQARNN, self).__init__()
    self.rnn = nn.RNN(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True,
    )
    self.fc = nn.Linear(hidden_size, output_size)

  def forward(self, x):
    out, _ = self.rnn(x)
    logits = self.fc(out)
    return logits

input_size = vocab_size
hidden_size = 128
num_layers = 1

model = SimpleQARNN(
    input_size=input_size,
    hidden_size=hidden_size,
    output_size=vocab_size,
    num_layers=num_layers,
)

criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

print("Початок навчання:\n")
for epoch in range(1, epochs + 1):
  model.train()
  optimizer.zero_grad()

  outputs = model(X_one_hot)

  loss = criterion(outputs.view(-1, vocab_size), Y_indices.view(-1))

  loss.backward()
  optimizer.step()

  print(f"Епоха [{epoch}/{epochs}] | Loss: {loss.item():.4f}")